# **Purpose**

**Task:** context + answer → question

This notebook fine-tunes `google/flan-t5-base` on a subset of SQuAD to generate a question given a context passage and an answer span. It's designed as a reusable template - swap `MODEL_NAME` and rerun to benchmark other Flan-T5 sizes.

**Runtime:** Runtime → Change runtime type → GPU (T4 is enough for `flan-t5-base` with the settings below).

## **Install dependencies**

In [1]:
!pip install -qU \
 transformers==5.16.1 \
 sentencepiece==0.2.2 \
 accelerate==1.14.0 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 47.4 MB/s eta 0:00:00


## **Imports**

In [2]:
import numpy as np
import torch
import transformers
import accelerate
import sentencepiece as spm

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

from kaggle_secrets import UserSecretsClient
import os

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("SentencePiece:", spm.__version__)

torch: 2.10.0+cu128
transformers: 5.16.1
accelerate: 1.14.0
SentencePiece: 0.2.2


## **Load the API Keys and Tokens**


In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")   # must match the exact secret name you set in Kaggle Secrets

os.environ["HF_TOKEN"] = hf_token

## **Config**

Tweak these for your experiment. `TRAIN_SUBSET_SIZE` / `VAL_SUBSET_SIZE` control how much of SQuAD you use - start small to sanity-check the pipeline before scaling up.

In [4]:
MODEL_NAME = "google/flan-t5-base"      # try "google/flan-t5-small" if GPU memory/time is tight
MAX_INPUT_LENGTH = 512                  # context+answer prompt length
MAX_TARGET_LENGTH = 96                  # questions are short
TRAIN_SUBSET_SIZE = 10000                # subset of SQuAD train split, set to None for full data
VAL_SUBSET_SIZE = 10
TRAIN_EPOCH_SIZE = 4
OUTPUT_DIR = "/content/flan-t5-base-qg"
SEED = 42

## **Load SQuAD and take a subset**

Uses the Hugging Face `squad` dataset. Swap to `"squad_v2"` if you also want unanswerable examples (note: `squad_v2` has empty answer lists for some examples, which the preprocessing below already handles gracefully).

In [5]:
raw = load_dataset("squad")

train_ds = raw["train"].shuffle(seed=SEED)
val_ds = raw["validation"].shuffle(seed=SEED)

if TRAIN_SUBSET_SIZE:
    train_ds = train_ds.select(range(TRAIN_SUBSET_SIZE))
if VAL_SUBSET_SIZE:
    val_ds = val_ds.select(range(VAL_SUBSET_SIZE))

print(train_ds)
print(val_ds)

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10000
})
Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 10
})


In [6]:
# Peek at one example
train_ds[0]

{'id': '573173d8497a881900248f0c',
 'title': 'Egypt',
 'context': 'The Pew Forum on Religion & Public Life ranks Egypt as the fifth worst country in the world for religious freedom. The United States Commission on International Religious Freedom, a bipartisan independent agency of the US government, has placed Egypt on its watch list of countries that require close monitoring due to the nature and extent of violations of religious freedom engaged in or tolerated by the government. According to a 2010 Pew Global Attitudes survey, 84% of Egyptians polled supported the death penalty for those who leave Islam; 77% supported whippings and cutting off of hands for theft and robbery; and 82% support stoning a person who commits adultery.',
 'question': 'What percentage of Egyptians polled support death penalty for those leaving Islam?',
 'answers': {'text': ['84%'], 'answer_start': [468]}}

## **Load tokenizer and model**

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## **Preprocessing**

Builds the prompt.

**Tip:** for the "answer highlighting" trick used in a lot of QG literature, wrap the answer span inside the context with a marker (e.g. `<hl> {answer} <hl>`) before building the prompt — this can measurably improve which part of the context the model attends to.

In [8]:
def preprocess(examples):
    inputs = []
    for context, answers in zip(examples["context"], examples["answers"]):
        answer_text = answers["text"][0] if len(answers["text"]) > 0 else ""
        prompt = (
            f"Target Answer: {answer_text}\n"
            f"Generate a question from the following context where the target answer is the correct answer. "
            f"Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\n"
            f"Context: {context}"
        )
        inputs.append(prompt)

    targets = examples["question"]

    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

## **Training arguments**

On Colab, set `fp16=True` if you have a T4/V100/A100 GPU (NVIDIA mixed precision). Adjust batch size / gradient accumulation if you hit out-of-memory errors.

In [9]:
training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    num_train_epochs=TRAIN_EPOCH_SIZE,
    fp16=torch.cuda.is_available(),   # mixed precision on Colab GPU
    logging_steps=50,
    save_total_limit=2,
    report_to="none",   # set to "wandb"/"tensorboard" if you use experiment tracking
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=data_collator,
)

## **Train the model**

In [10]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,6.570513
100,6.507958
150,6.467363
200,6.367360
250,6.217914
300,6.364173
350,5.367734
400,5.084010
450,5.077825
500,5.238007


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1252, training_loss=4.893583018558855, metrics={'train_runtime': 2854.8054, 'train_samples_per_second': 14.011, 'train_steps_per_second': 0.439, 'total_flos': 2.0839969460404224e+16, 'train_loss': 4.893583018558855, 'epoch': 4.0})

## **Save the fine-tuned model**

In [11]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/flan-t5-base-qg


## **Push the model to Huggingface Hub**

In [12]:
model.push_to_hub("gaurav-dey/flan-t5-base-qg")
tokenizer.push_to_hub("gaurav-dey/flan-t5-base-qg")

# model.push_to_hub("gauravdey2024/flan-t5-base-qg")
# tokenizer.push_to_hub("gauravdey2024/flan-t5-base-qg")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/gaurav-dey/flan-t5-base-qg/commit/f4e290290426fcb22e04b1bb90bbea212f2688ee', commit_message='Upload tokenizer', commit_description='', oid='f4e290290426fcb22e04b1bb90bbea212f2688ee', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gaurav-dey/flan-t5-base-qg', endpoint='https://huggingface.co', repo_type='model', repo_id='gaurav-dey/flan-t5-base-qg'), pr_revision=None, pr_num=None)

Optional: mount Google Drive and copy the checkpoint there so it persists after the Colab runtime disconnects.

## **Sanity-check generations**

In [13]:
sample = val_ds.select(range(5))
for ex in sample:
    answer_text = ex["answers"]["text"][0] if ex["answers"]["text"] else ""

    prompt = (
        f"Target Answer: {answer_text}\n"
        f"Generate a question from the following context where the target answer is the correct answer. "
        f"Do not include phrases like 'According to the text' in the question and do not repeat the context in the question.\n"
        f"Context: {ex['context']}"
    )

    input_ids = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LENGTH
    ).input_ids.to(model.device)

    output_ids = model.generate(input_ids, max_length=MAX_TARGET_LENGTH, num_beams=4)
    generated_question = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    print(f"Answer:              {answer_text}")
    print(f"Gold question:       {ex['question']}")
    print(f"Generated question:  {generated_question}")
    print("-" * 80)

Answer:              1852
Gold question:       In what year did Massachusetts first require children to be educated in schools?
Generated question:  When was compulsory education established in Massachusetts?
--------------------------------------------------------------------------------
Answer:              1962
Gold question:       When were stromules discovered?
Generated question:  When were stromules first observed in chloroplasts?
--------------------------------------------------------------------------------
Answer:              Horace Walpole
Gold question:       Which artist who had a major influence on the Gothic Revival is represented in the V&A's British galleries?
Generated question:  What Gothic Revival patron had an influence on the Gothic Revival?
--------------------------------------------------------------------------------
Answer:              several regional colleges and universities
Gold question:       In 1890, who did the university decide to team up with?
Ge

## **Next steps**

- Swap `MODEL_NAME` to `google/flan-t5-small` or `google/flan-t5-large` and rerun to compare sizes.
- Add the answer-highlighting (`<hl>` marker) preprocessing variant and compare against this baseline.
- Add a round-trip QA-consistency evaluation metric for a semantic-quality signal beyond ROUGE/BLEU.